In [1]:
# ================================
# exp26_xgboost_regression
# XGBoost regression model
# ================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor


# ================================
# Metric diagnostics tool
# ================================

def metric_diagnostics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(np.maximum(y_pred,0))
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    nrmse_mean = rmse / np.mean(y_true)
    nrmse_range = rmse / (np.max(y_true) - np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("RMSLE:", rmsle)
    print("NRMSE (mean):", nrmse_mean)
    print("NRMSE (range):", nrmse_range)


# ================================
# Load data
# ================================

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

target = "含水率"
id_col = "sample number"

spectral_cols = [
    c for c in train.columns
    if c not in ["sample number","species number","樹種","含水率"]
]

X = train[spectral_cols]
y = train[target]

X_test = test[spectral_cols]


# ================================
# XGBoost configurations
# ================================

xgb_configs = [

    {"max_depth":4, "learning_rate":0.05},
    {"max_depth":5, "learning_rate":0.05},
    {"max_depth":4, "learning_rate":0.03}

]


# ================================
# KFold
# ================================

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_predictions = np.zeros(len(X))
test_predictions = np.zeros(len(X_test))


# ================================
# Training loop
# ================================

for params in xgb_configs:

    print(f"\nTraining XGBoost {params}")

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))

    model = XGBRegressor(

        n_estimators=800,
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model.fit(X_train, y_train)

        pred_val = model.predict(X_val)

        oof[val_idx] = pred_val

        test_pred += model.predict(X_test) / kf.n_splits

    metric_diagnostics(y, oof)

    oof_predictions += oof / len(xgb_configs)
    test_predictions += test_pred / len(xgb_configs)


# ================================
# Final diagnostics
# ================================

print("\nFinal Ensemble Performance")
metric_diagnostics(y, oof_predictions)


# ================================
# Save submission
# ================================

os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    id_col: test[id_col],
    target: test_predictions
})

output_path = "../submissions/exp26_xgboost_regression.csv"

submission.to_csv(output_path, index=False, header=False)

print("\nSubmission saved:", output_path)


Training XGBoost {'max_depth': 4, 'learning_rate': 0.05}

Metric diagnostics
------------------
RMSE: 14.891197877970976
MAE: 6.535393158733792
RMSLE: 0.21906344789114882
NRMSE (mean): 0.29819567615403486
NRMSE (range): 0.05001389350662665

Training XGBoost {'max_depth': 5, 'learning_rate': 0.05}

Metric diagnostics
------------------
RMSE: 14.889399521986922
MAE: 6.380755234929181
RMSLE: 0.20463704789412038
NRMSE (mean): 0.2981596641432467
NRMSE (range): 0.050007853510018546

Training XGBoost {'max_depth': 4, 'learning_rate': 0.03}

Metric diagnostics
------------------
RMSE: 15.065557869495859
MAE: 6.863616460912715
RMSLE: 0.22319524860057482
NRMSE (mean): 0.30168722841148554
NRMSE (range): 0.050599502677856734

Final Ensemble Performance

Metric diagnostics
------------------
RMSE: 14.857611718549373
MAE: 6.5142895265686445
RMSLE: 0.2118257499410663
NRMSE (mean): 0.29752311457771213
NRMSE (range): 0.049901090318167675

Submission saved: ../submissions/exp26_xgboost_regression.csv
